# 00 – Setup check

**What:** Check that the `astro-pipeline` environment is complete and that MAST can be reached.

**Why:** Before downloading data, we want to know that all packages import and the archive responds – like a test scan with a calibration slide before the real sample goes on the microscope.

**Result:** The astro-pipeline environment works (Python 3.12.16, astropy 8.0.2, astroquery 0.4.11, reproject 0.21.0, numpy 2.5.4, matplotlib 3.11.2). MAST is reachable: programme 2733 has 10 stage-3 observations (6 NIRCam, 4 MIRI).

## 1. Python and package versions

The printed versions go into `docs/worklog.md` so that the project can be reproduced later.

In [1]:
import sys
import numpy, matplotlib, astropy, astroquery, reproject

print("python    ", sys.version.split()[0])
print("executable", sys.executable)  # should point into .../envs/astro-pipeline/
for mod in (numpy, matplotlib, astropy, astroquery, reproject):
    print(f"{mod.__name__:<10}", mod.__version__)

python     3.12.16
executable /Users/inu/miniforge3/envs/astro-pipeline/bin/python
numpy      2.5.4
matplotlib 3.11.2
astropy    8.0.2
astroquery 0.4.11
reproject  0.21.0


## 2. Is the notebook running in the right environment?

If `executable` above does **not** contain `envs/astro-pipeline`, the wrong kernel is running (e.g. `bia` or base). Then select the `astro-pipeline` kernel at the top right in JupyterLab.

## 3. Is MAST reachable?

A small metadata query, **no download**: which observations exist for programme 2733 (NGC 3132)? Just a foretaste of stage 1.

Note: I set column names and values (e.g. `instrument_name`) to the best of my knowledge – if the query comes back empty, we drop the instrument filter and look at the columns.

In [2]:
from astroquery.mast import Observations

obs = Observations.query_criteria(
    obs_collection="JWST",
    proposal_id="2733",
)
print(len(obs), "observations found")
obs["instrument_name", "filters", "calib_level", "t_exptime"]

10 observations found


instrument_name,filters,calib_level,t_exptime
str12,str11,int64,float64
MIRI/IMAGE,F1130W,3,2708.4320000000002
MIRI/IMAGE,F770W,3,2708.4320000000002
MIRI/IMAGE,F1800W,3,2708.4320000000002
MIRI/IMAGE,F1280W,3,2708.4320000000002
NIRCAM/IMAGE,F187N,3,2319.144
NIRCAM/IMAGE,F090W,3,1460.2000000000003
NIRCAM/IMAGE,F356W,3,1460.2000000000003
NIRCAM/IMAGE,F212N,3,2319.144
NIRCAM/IMAGE,F444W;F405N,3,2319.144


## Conclusion

All packages import, and the kernel uses the right environment (`envs/astro-pipeline`). The NIRCam filters (F090W, F187N, F212N, F356W, F405N, F470N) match those of the official NASA image; MIRI provides F770W, F1130W, F1280W, F1800W. F405N and F470N appear in combination with F444W (`F444W;F405N`) – we check this in the header in stage 2. The narrow-band filters were exposed longer (≈2320 s instead of ≈1460 s) because less light passes through them.